# System Resource Monitor

This notebook samples Windows system resources on a fixed interval for a fixed total
duration and writes the results to two Excel workbooks: a dashboard that always holds
the latest snapshot and a log that grows for the full run. Every row on every sheet of
both files carries the sample timestamp.

## Config

This is the only cell you adjust. It holds the sample interval, the total run duration,
and the two output file paths. Every later section reads from these values, so changing
them here changes the whole run. The defaults are a 5 minute interval over 24 hours,
which is 288 samples.

In [7]:
from datetime import timedelta
from pathlib import Path

# How long to wait between samples.
SAMPLE_INTERVAL = timedelta(minutes=5)

# How long the whole run lasts. After this elapses the loop stops.
TOTAL_DURATION = timedelta(hours=24)

# Output file paths. Do not change these names.
DASHBOARD_PATH = Path("outputs") / "dashboard" / "dashboard.xlsx"
LOG_PATH = Path("outputs") / "log" / "log.xlsx"

# Make sure the output folders exist.
DASHBOARD_PATH.parent.mkdir(parents=True, exist_ok=True)
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)

# Derived: total number of samples this run will take.
SAMPLE_COUNT = int(TOTAL_DURATION / SAMPLE_INTERVAL)

print("Sample interval :", SAMPLE_INTERVAL)
print("Total duration  :", TOTAL_DURATION)
print("Planned samples :", SAMPLE_COUNT)
print("Dashboard file  :", DASHBOARD_PATH)
print("Log file        :", LOG_PATH)

Sample interval : 0:05:00
Total duration  : 1 day, 0:00:00
Planned samples : 288
Dashboard file  : outputs\dashboard\dashboard.xlsx
Log file        : outputs\log\log.xlsx


## Dependencies setup

This section makes the notebook run on a fresh machine that has nothing installed beyond
Python itself. It checks for each package the notebook needs and installs or upgrades only
what is missing or too old, so running it twice does no harm. The packages are `psutil`
for reading system resources and `openpyxl` for writing the Excel files.

A minimum version is enforced for each package, not just its presence. This matters
because some bundled Python setups (Anaconda for example) ship an old `psutil` that fails
on Python 3.12 when reading disks. The cell upgrades those automatically.

Two notes:

- Python 3 must already be installed, because the notebook needs Python to run at all. If
  you are on a machine with no Python, install Python 3 from python.org first, then open
  this notebook.
- If this cell reports that it upgraded or installed anything, restart the kernel and run
  the notebook again from the top, so the new version is the one actually loaded.


In [8]:
import importlib
import importlib.metadata as metadata
import subprocess
import sys

# Import name mapped to its pip name and the minimum version the notebook needs.
REQUIRED_PACKAGES = {
    "psutil": {"pip": "psutil", "min": "5.9.6"},
    "openpyxl": {"pip": "openpyxl", "min": "3.0.0"},
}


def _version_tuple(text):
    """Turn a version string like '5.9.6' into a comparable tuple of integers."""
    parts = []
    for chunk in text.split("."):
        digits = ""
        for char in chunk:
            if char.isdigit():
                digits += char
            else:
                break
        parts.append(int(digits) if digits else 0)
    return tuple(parts)


def _pip_install(spec):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", spec])


def ensure_packages(packages):
    """Install or upgrade each package so it is present and meets the minimum version."""
    print("Python version:", sys.version.split()[0])
    print("-" * 50)

    changed = False
    for import_name, info in packages.items():
        pip_name = info["pip"]
        minimum = info["min"]
        try:
            current = metadata.version(pip_name)
        except metadata.PackageNotFoundError:
            current = None

        if current is None:
            print("NOT INSTALLED  :", pip_name, "(need >=", minimum + ")")
            _pip_install(pip_name + ">=" + minimum)
            changed = True
        elif _version_tuple(current) < _version_tuple(minimum):
            print("OUTDATED       :", pip_name, current, "(need >=", minimum + ")")
            _pip_install(pip_name + ">=" + minimum)
            changed = True
        else:
            print("OK             :", pip_name, current)

    print("-" * 50)
    if changed:
        print("Packages were installed or upgraded.")
        print("Restart the kernel and run the notebook again from the top.")
    else:
        print("All required packages are present and up to date.")


ensure_packages(REQUIRED_PACKAGES)


Python version: 3.12.7
--------------------------------------------------
OK             : psutil 7.2.2
OK             : openpyxl 3.1.5
--------------------------------------------------
All required packages are present and up to date.


## CPU usage

This section reads CPU load. It returns the overall busy percentage and the busy
percentage of each logical core. The reading uses a one second measurement window
(`interval=1`) so the value reflects real activity rather than the zero you get from an
instant call. The overall figure is the mean of the per-core figures, so the two always
agree. The result is a flat dictionary of named fields plus the sample timestamp, which
is the shape every system-metrics collector in this notebook returns.

In [9]:
import psutil
from datetime import datetime


def collect_cpu(timestamp):
    """Return overall and per-core CPU usage for one sample, in plain labels."""
    per_core = psutil.cpu_percent(interval=1, percpu=True)
    overall = round(sum(per_core) / len(per_core), 1) if per_core else 0.0

    row = {
        "Time": timestamp,
        "Overall CPU Usage (%)": overall,
        "Number of CPU Cores": len(per_core),
    }
    for index, value in enumerate(per_core):
        # Cores numbered from 1 so the labels read naturally.
        row["Core {} Usage (%)".format(index + 1)] = value
    return row


# Quick check.
collect_cpu(datetime.now())


{'Time': datetime.datetime(2026, 6, 10, 14, 38, 44, 786273),
 'Overall CPU Usage (%)': 36.5,
 'Number of CPU Cores': 8,
 'Core 1 Usage (%)': 33.8,
 'Core 2 Usage (%)': 34.4,
 'Core 3 Usage (%)': 71.2,
 'Core 4 Usage (%)': 40.6,
 'Core 5 Usage (%)': 28.1,
 'Core 6 Usage (%)': 28.1,
 'Core 7 Usage (%)': 25.0,
 'Core 8 Usage (%)': 31.2}

## CPU temperature

This section reads the CPU thermal sensor. Windows does not expose CPU temperature
through psutil, so the reading is pulled from the WMI thermal zone using a short
PowerShell call. WMI reports the value in tenths of a Kelvin, which this code converts to
degrees Celsius. Many laptops block this sensor or return access denied. When that
happens the field is recorded as `Unavailable` for that sample rather than letting the
error stop the run, so the timestamp and the rest of the sample are still captured.

In [10]:
import subprocess
from datetime import datetime


def _run_powershell(command, timeout=15):
    """Run a PowerShell command and return its text output, or None on failure."""
    try:
        result = subprocess.run(
            ["powershell", "-NoProfile", "-NonInteractive", "-Command", command],
            capture_output=True,
            text=True,
            timeout=timeout,
        )
        if result.returncode == 0:
            return result.stdout.strip()
    except Exception:
        pass
    return None


def collect_temperature(timestamp):
    """Return the CPU temperature in Celsius, or Unavailable if the sensor is blocked."""
    command = (
        "Get-CimInstance -Namespace root/wmi "
        "-ClassName MSAcpi_ThermalZoneTemperature -ErrorAction Stop "
        "| Select-Object -ExpandProperty CurrentTemperature"
    )
    output = _run_powershell(command)

    temperature = "Unavailable"
    if output:
        readings = []
        for line in output.splitlines():
            line = line.strip()
            if line.isdigit():
                # WMI reports tenths of a Kelvin. Convert to Celsius.
                readings.append(int(line) / 10.0 - 273.15)
        if readings:
            temperature = round(sum(readings) / len(readings), 1)

    return {
        "Time": timestamp,
        "CPU Temperature (°C)": temperature,
    }


# Quick check.
collect_temperature(datetime.now())

{'Time': datetime.datetime(2026, 6, 10, 14, 38, 45, 801902),
 'CPU Temperature (°C)': 'Unavailable'}

## RAM

This section reads system memory. It reports the total memory fitted, how much is in use,
how much is still available, and the used percentage. The raw figures from the system are
in bytes, which are hard to read, so they are converted to gigabytes (GB) and rounded.
Available memory is the amount that can be handed to programs without swapping, which is
the number most people care about when asking how much memory is free.

In [11]:
import psutil
from datetime import datetime

BYTES_PER_GB = 1024 ** 3


def to_gb(value):
    """Convert a byte count to gigabytes, rounded to two decimals."""
    return round(value / BYTES_PER_GB, 2)


def collect_ram(timestamp):
    """Return total, used, available memory in GB and the used percentage."""
    memory = psutil.virtual_memory()
    return {
        "Time": timestamp,
        "Total Memory (GB)": to_gb(memory.total),
        "Used Memory (GB)": to_gb(memory.used),
        "Available Memory (GB)": to_gb(memory.available),
        "Memory Usage (%)": memory.percent,
    }


# Quick check.
collect_ram(datetime.now())

{'Time': datetime.datetime(2026, 6, 10, 14, 38, 46, 252331),
 'Total Memory (GB)': 7.71,
 'Used Memory (GB)': 7.16,
 'Available Memory (GB)': 0.55,
 'Memory Usage (%)': 92.9}

## Disk

This section reads storage usage for every drive on the machine. Unlike the single
system-metrics row, disk returns one row per drive, so it gets its own sheet in both
Excel files. Each row carries the sample timestamp, the drive letter, total size, used
space, free space (all in gigabytes), and the used percentage. Some entries such as an
empty CD drive or card reader raise an error when read. Those are skipped so they do not
stop the sample, and the drives that do respond are still recorded.

In [12]:
import psutil
from datetime import datetime

BYTES_PER_GB = 1024 ** 3


def _gb(value):
    return round(value / BYTES_PER_GB, 2)


def collect_disk(timestamp):
    """Return one row per drive with total, used, free space and used percentage."""
    rows = []
    for partition in psutil.disk_partitions(all=False):
        try:
            usage = psutil.disk_usage(partition.mountpoint)
        except Exception:
            # Empty CD drive, card reader, or a drive psutil cannot read on Windows
            # (this can raise PermissionError, OSError, or SystemError). Skip it so
            # one bad drive does not stop the sample.
            continue
        rows.append({
            "Time": timestamp,
            "Drive": partition.device,
            "Total Space (GB)": _gb(usage.total),
            "Used Space (GB)": _gb(usage.used),
            "Free Space (GB)": _gb(usage.free),
            "Disk Usage (%)": usage.percent,
        })
    return rows


# Quick check.
collect_disk(datetime.now())


[{'Time': datetime.datetime(2026, 6, 10, 14, 38, 46, 271425),
  'Drive': 'C:\\',
  'Total Space (GB)': 476.23,
  'Used Space (GB)': 352.82,
  'Free Space (GB)': 123.41,
  'Disk Usage (%)': 74.1},
 {'Time': datetime.datetime(2026, 6, 10, 14, 38, 46, 271425),
  'Drive': 'D:\\',
  'Total Space (GB)': 237.86,
  'Used Space (GB)': 72.83,
  'Free Space (GB)': 165.03,
  'Disk Usage (%)': 30.6}]

## Battery

This section reads the battery. It reports the charge percentage, whether the laptop is
plugged in, and the estimated time left on battery. The estimate is given by the system in
seconds, which is converted here to a plain reading like `2 hours 15 minutes`. When the
machine is plugged in or the system cannot estimate the time, that is stated in words
rather than shown as a confusing number. On a desktop with no battery the fields read
`No battery` so the row still records cleanly with its timestamp.

In [13]:
import psutil
from datetime import datetime


def _format_time_left(seconds):
    """Turn a seconds-remaining value into a plain reading."""
    if seconds == psutil.POWER_TIME_UNLIMITED:
        return "Plugged in"
    if seconds == psutil.POWER_TIME_UNKNOWN or seconds is None or seconds < 0:
        return "Unknown"
    hours = seconds // 3600
    minutes = (seconds % 3600) // 60
    return "{} hours {} minutes".format(hours, minutes)


def collect_battery(timestamp):
    """Return battery charge, plugged-in state and estimated time remaining."""
    battery = psutil.sensors_battery()
    if battery is None:
        return {
            "Time": timestamp,
            "Battery Charge (%)": "No battery",
            "Power Plugged In": "No battery",
            "Estimated Time Remaining": "No battery",
        }
    return {
        "Time": timestamp,
        "Battery Charge (%)": round(battery.percent, 1),
        "Power Plugged In": "Yes" if battery.power_plugged else "No",
        "Estimated Time Remaining": (
            "Plugged in" if battery.power_plugged
            else _format_time_left(battery.secsleft)
        ),
    }


# Quick check.
collect_battery(datetime.now())

{'Time': datetime.datetime(2026, 6, 10, 14, 38, 46, 288310),
 'Battery Charge (%)': 49,
 'Power Plugged In': 'No',
 'Estimated Time Remaining': '0 hours 42 minutes'}